# Bayesian semantic mapping

**Track D · Scene & World Models** · maps to lesson D4 (Bayesian label fusion).

A map is built from many **noisy** observations. We fuse them the principled way: **log-odds** for occupancy and **count-based posteriors** for the semantic class, and watch the map converge to the truth.

> CPU is fine.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
Gh, Gw, Kc = 32, 32, 4
STEPS = int(os.environ.get("STEPS", 300))
gt_label = rng.integers(0, Kc, size=(Gh, Gw))
gt_occ = (rng.random((Gh, Gw)) < 0.45)

## 1 · Sensor model — observations are right with prob. p

In [ ]:
p_occ, p_lab = 0.8, 0.7
logodds = np.zeros((Gh, Gw)); counts = np.ones((Gh, Gw, Kc))   # Dirichlet prior
def observe():
    ys = rng.integers(0, Gh, 60); xs = rng.integers(0, Gw, 60)   # a sensor footprint
    for y, x in zip(ys, xs):
        z_occ = gt_occ[y, x] if rng.random() < p_occ else not gt_occ[y, x]
        logodds[y, x] += (1.84 if z_occ else -1.84)             # log(p/(1-p))
        z_lab = gt_label[y, x] if rng.random() < p_lab else rng.integers(0, Kc)
        counts[y, x, z_lab] += 1

## 2 · Fuse over time, tracking accuracy

In [ ]:
hist = []
for step in range(STEPS + 1):
    observe()
    if step % max(1, STEPS // 10) == 0:
        occ_acc = ((logodds > 0) == gt_occ).mean()
        lab_acc = (counts.argmax(-1) == gt_label).mean()
        hist.append((step, round(float(occ_acc), 3), round(float(lab_acc), 3)))
        print(f"step {step:4d}  occupancy acc {occ_acc:.3f}  label acc {lab_acc:.3f}")

## 3 · Compare — recovered maps vs. ground truth

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(7, 7))
ax[0, 0].imshow(gt_occ); ax[0, 0].set_title("true occupancy"); ax[0, 1].imshow(logodds > 0); ax[0, 1].set_title("estimated")
ax[1, 0].imshow(gt_label, cmap="tab10"); ax[1, 0].set_title("true labels"); ax[1, 1].imshow(counts.argmax(-1), cmap="tab10"); ax[1, 1].set_title("estimated")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/D_semantic_mapping/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/D_semantic_mapping"; os.makedirs(run, exist_ok=True)
json.dump({"history": hist}, open(f"{run}/metrics.json", "w"), indent=2)
np.save(f"{run}/occ.npy", (logodds > 0)); np.save(f"{run}/labels.npy", counts.argmax(-1))
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Combines **C** perception with **LM** open-vocabulary labels.

### Where to go next
- Replace random footprints with a moving camera frustum and real per-pixel class probabilities (e.g. from the CLIP / DINOv2 labs) → **open-vocabulary mapping**.
- Add a 3D voxel grid and fuse into the TSDF for a semantic 3D map.